In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import requests
from bs4 import BeautifulSoup
import time
import re
from google.colab import drive


# --- 2. CẤU HÌNH ---
TARGET_URL = "https://en.wikipedia.org/wiki/Hanoi"
# Đường dẫn lưu trên Drive (Thư mục Wiki_Downloads sẽ tự tạo trong Drive của bạn)
ROOT_DIR = "/content/drive/MyDrive/Wiki_Downloads_Hanoi"

# QUAN TRỌNG: Wikipedia yêu cầu thông tin định danh hợp lệ để tránh lỗi 429
HEADERS = {
    'User-Agent': 'ColabBot/1.0 (contact: yourname@gmail.com) Python-Requests-Scraper'
}

def get_safe_large_url(url):
    """Lấy bản thumbnail 800px thay vì ảnh gốc để tránh bị chặn IP"""
    if not url: return None
    full_url = 'https:' + url if url.startswith('//') else url

    if '/thumb/' in full_url:
        # Sử dụng Regex để thay thế kích thước thumbnail thành 800px
        full_url = re.sub(r'/\d+px-', '/800px-', full_url)
    return full_url

def download_on_colab():
    if not os.path.exists(ROOT_DIR):
        os.makedirs(ROOT_DIR)
        print(f"📁 Đã tạo thư mục: {ROOT_DIR}")

    try:
        print(f"🌐 Đang truy cập: {TARGET_URL}")
        res = requests.get(TARGET_URL, headers=HEADERS, timeout=15)
        res.raise_for_status() # Kiểm tra nếu lỗi kết nối

        soup = BeautifulSoup(res.text, 'html.parser')
        content = soup.find(id="mw-content-text")
        all_imgs = content.find_all('img')

        print(f"📸 Tìm thấy {len(all_imgs)} thẻ ảnh. Bắt đầu lọc và tải...")

        count = 0
        for img in all_imgs:
            # Lọc ảnh qua thuộc tính width (Wikipedia thường set sẵn)
            try:
                w = int(img.get('width', 0))
            except: w = 0

            if w < 60: continue # Bỏ qua các icon nhỏ

            img_url = get_safe_large_url(img.get('src'))

            try:
                # Tăng time.sleep lên 3 giây vì Colab dùng chung IP với nhiều người,
                # dễ bị Wikipedia đánh dấu là bot hơn
                time.sleep(1)

                img_res = requests.get(img_url, headers=HEADERS, timeout=15)

                if img_res.status_code == 200:
                    # Kiểm tra xem có phải file HTML lỗi không (như bạn đã bị)
                    if "text/html" in img_res.headers.get('Content-Type', ''):
                        print(f"⚠️ Bỏ qua ảnh {count} vì server trả về HTML lỗi 429.")
                        continue

                    ext = img_url.split('.')[-1].lower()[:3]
                    if ext not in ['jpg', 'png', 'web', 'jpe']: ext = 'jpg'

                    filename = f"hanoi_img_{count}.{ext}"
                    file_path = os.path.join(ROOT_DIR, filename)

                    with open(file_path, 'wb') as f:
                        f.write(img_res.content)

                    print(f"✅ [{count}] Đã tải: {filename}")
                    count += 1

                elif img_res.status_code == 429:
                    print("🚫 Lỗi 429: Quá nhiều yêu cầu. Hãy dừng lại và đổi User-Agent hoặc đợi 30 phút.")
                    break

            except Exception as e:
                print(f"❌ Lỗi khi tải ảnh {count}: {e}")
                continue

        print(f"\n✨ HOÀN TẤT! Tổng cộng {count} ảnh đã được lưu vào Google Drive.")

    except Exception as e:
        print(f"💥 Lỗi hệ thống: {e}")

# Chạy hàm tải
if __name__ == "__main__":
    download_on_colab()

🌐 Đang truy cập: https://en.wikipedia.org/wiki/Hanoi
📸 Tìm thấy 91 thẻ ảnh. Bắt đầu lọc và tải...
✅ [0] Đã tải: hanoi_img_0.jpg
✅ [1] Đã tải: hanoi_img_1.jpg
✅ [2] Đã tải: hanoi_img_2.jpg
✅ [3] Đã tải: hanoi_img_3.jpg
✅ [4] Đã tải: hanoi_img_4.jpg
✅ [5] Đã tải: hanoi_img_5.jpg
✅ [6] Đã tải: hanoi_img_6.jpg
✅ [7] Đã tải: hanoi_img_7.jpg
✅ [8] Đã tải: hanoi_img_8.png
✅ [9] Đã tải: hanoi_img_9.png
✅ [10] Đã tải: hanoi_img_10.jpg
✅ [11] Đã tải: hanoi_img_11.jpg
✅ [12] Đã tải: hanoi_img_12.jpg
✅ [13] Đã tải: hanoi_img_13.png
✅ [14] Đã tải: hanoi_img_14.png
✅ [15] Đã tải: hanoi_img_15.jpg
✅ [16] Đã tải: hanoi_img_16.jpg
✅ [17] Đã tải: hanoi_img_17.jpg
✅ [18] Đã tải: hanoi_img_18.jpg
✅ [19] Đã tải: hanoi_img_19.jpg
✅ [20] Đã tải: hanoi_img_20.jpg
✅ [21] Đã tải: hanoi_img_21.jpg
✅ [22] Đã tải: hanoi_img_22.jpg
✅ [23] Đã tải: hanoi_img_23.jpg
✅ [24] Đã tải: hanoi_img_24.jpg
✅ [25] Đã tải: hanoi_img_25.jpg
✅ [26] Đã tải: hanoi_img_26.jpg
✅ [27] Đã tải: hanoi_img_27.jpg
✅ [28] Đã tải: hanoi_img_2

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
import json
import time
import re
from google.colab import drive


# --- 2. CẤU HÌNH ---
JSON_FOLDER = "/content/drive/MyDrive/wiki_data_json_final"
ROOT_SAVE_DIR = "/content/drive/MyDrive/Wiki_Deep_Images"
MIN_SIZE = 50

# Header có email giúp Wikipedia "nhẹ tay" hơn với bạn
HEADERS = {
    'User-Agent': 'WikiExplorer/5.0 (contact: yourname@email.com) DataGatheringBot'
}

def clean_name(text):
    if not text: return "Unknown"
    return re.sub(r'[\\/*?:"<>|]', '', text).strip().replace(' ', '_')

def get_full_res_url(url):
    if not url: return None
    full_url = 'https:' + url if url.startswith('//') else url
    if '/thumb/' in full_url:
        parts = full_url.replace('/thumb/', '/').split('/')
        return '/'.join(parts[:-1])
    return full_url

def is_valid_image(response):
    """Xác thực thực sự là ảnh dựa trên Headers và Content"""
    content_type = response.headers.get('Content-Type', '')
    if 'image' not in content_type:
        return False
    # Nếu nội dung chứa thẻ html thì chắc chắn là file lỗi 429
    if b"<!DOCTYPE html" in response.content[:100]:
        return False
    return True

def start_mining():
    if not os.path.exists(ROOT_SAVE_DIR): os.makedirs(ROOT_SAVE_DIR)

    json_files = sorted([f for f in os.listdir(JSON_FOLDER) if f.endswith('.json')])

    for j_file in json_files:
        print(f"\n📂 File JSON: {j_file}")
        with open(os.path.join(JSON_FOLDER, j_file), 'r', encoding='utf-8') as f:
            try: data = json.load(f)
            except: continue

        for item in data:
            country = clean_name(item.get('country', 'UnknownCountry'))
            title = clean_name(item.get('title', 'UnknownPlace'))
            wiki_url = item.get('url')

            if not wiki_url: continue

            country_dir = os.path.join(ROOT_SAVE_DIR, country)
            os.makedirs(country_dir, exist_ok=True)

            # --- CƠ CHẾ SKIP ---
            # Nếu đã có file ảnh đầu tiên của địa danh này, coi như xong bài này
            first_img_check = os.path.join(country_dir, f"{country}_{title}_0.jpg")
            if os.path.exists(first_img_check):
                continue

            print(f"🚀 Vét ảnh: {title} ({country})")

            try:
                time.sleep(1.5) # Nghỉ để server hồi phục
                res = requests.get(wiki_url, headers=HEADERS, timeout=15)
                if res.status_code != 200: continue

                soup = BeautifulSoup(res.text, 'html.parser')
                content = soup.find(id="mw-content-text")
                if not content: continue

                imgs = content.find_all('img')
                downloaded_urls = set()
                count = 0

                for img in imgs:
                    try:
                        w, h = int(img.get('width', 0)), int(img.get('height', 0))
                        if w < MIN_SIZE and h < MIN_SIZE: continue

                        img_url = get_full_res_url(img.get('src'))
                        if not img_url or img_url in downloaded_urls: continue

                        # Tải và xác thực
                        time.sleep(0.8)
                        img_res = requests.get(img_url, headers=HEADERS, timeout=15)

                        if img_res.status_code == 200 and is_valid_image(img_res):
                            ext = img_url.split('.')[-1].lower().split('?')[0]
                            if len(ext) > 4: ext = 'jpg'

                            filename = f"{country}_{title}_{count}.{ext}"
                            with open(os.path.join(country_dir, filename), 'wb') as f:
                                f.write(img_res.content)

                            downloaded_urls.add(img_url)
                            count += 1
                        elif img_res.status_code == 429:
                            print("⚠️ Bị chặn 429! Tạm nghỉ 2 phút...")
                            time.sleep(120)
                            break # Thoát bài hiện tại để chờ
                    except: continue

                if count > 0: print(f"   ✅ Đã lấy {count} ảnh.")

            except Exception as e:
                print(f"   ❌ Lỗi: {e}")
                continue

if __name__ == "__main__":
    start_mining()


📂 File JSON: data_1.json
🚀 Vét ảnh: Resistencia (Argentina)
🚀 Vét ảnh: Retiro (Argentina)


KeyboardInterrupt: 

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
import json
import time
import re
from google.colab import drive

# --- 1. KẾT NỐI DRIVE ---
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# --- 2. CẤU HÌNH ---
JSON_FOLDER = "/content/drive/MyDrive/wiki_data_json_final"
ROOT_SAVE_DIR = "/content/drive/MyDrive/Wiki_Deep_Images"
LOG_FILE = "/content/drive/MyDrive/Wiki_Deep_Images/processed_log.txt"
MIN_SIZE = 50

# Danh sách các quốc gia đã cào xong (Dựa trên ảnh bạn gửi)
SKIP_COUNTRIES = {
    "Albania", "Angola", "Argentina", "Armenia",
    "Belgium", "Malaysia", "Mexico", "Mozambique"
}

HEADERS = {
    'User-Agent': 'WikiExplorer/5.0 (contact: yourname@email.com) DataGatheringBot'
}

def clean_name(text):
    if not text: return "Unknown"
    return re.sub(r'[\\/*?:"<>|]', '', text).strip().replace(' ', '_')

def get_full_res_url(url):
    if not url: return None
    full_url = 'https:' + url if url.startswith('//') else url
    if '/thumb/' in full_url:
        parts = full_url.replace('/thumb/', '/').split('/')
        return '/'.join(parts[:-1])
    return full_url

def is_valid_image(response):
    content_type = response.headers.get('Content-Type', '')
    if 'image' not in content_type:
        return False
    if b"<!DOCTYPE html" in response.content[:100]:
        return False
    return True

def start_mining():
    if not os.path.exists(ROOT_SAVE_DIR): os.makedirs(ROOT_SAVE_DIR)

    # Đọc log để bỏ qua các URL lẻ đã hoàn thành
    processed_urls = set()
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, 'r', encoding='utf-8') as f:
            processed_urls = set(line.strip() for line in f)

    json_files = sorted([f for f in os.listdir(JSON_FOLDER) if f.endswith('.json')])

    for j_file in json_files:
        print(f"\n📂 Đang xử lý file: {j_file}")
        with open(os.path.join(JSON_FOLDER, j_file), 'r', encoding='utf-8') as f:
            try: data = json.load(f)
            except: continue

        for item in data:
            country_raw = item.get('country', 'Unknown')
            country = clean_name(country_raw)

            # --- KIỂM TRA QUỐC GIA ĐÃ XONG ---
            if country in SKIP_COUNTRIES:
                continue

            title = clean_name(item.get('title', 'UnknownPlace'))
            wiki_url = item.get('url')

            if not wiki_url or wiki_url in processed_urls:
                continue

            country_dir = os.path.join(ROOT_SAVE_DIR, country)
            os.makedirs(country_dir, exist_ok=True)

            # Check file thực tế đề phòng log chưa ghi
            if os.path.exists(os.path.join(country_dir, f"{country}_{title}_0.jpg")):
                continue

            print(f"🚀 Vét ảnh: {title} ({country})")

            try:
                time.sleep(1.5)
                res = requests.get(wiki_url, headers=HEADERS, timeout=15)
                if res.status_code == 429:
                    print("⚠️ Bị chặn 429! Nghỉ 3 phút...")
                    time.sleep(180)
                    continue
                if res.status_code != 200: continue

                soup = BeautifulSoup(res.text, 'html.parser')
                content = soup.find(id="mw-content-text")
                if not content: continue

                imgs = content.find_all('img')
                downloaded_urls = set()
                count = 0

                for img in imgs:
                    try:
                        w, h = int(img.get('width', 0)), int(img.get('height', 0))
                        if w < MIN_SIZE and h < MIN_SIZE: continue

                        img_url = get_full_res_url(img.get('src'))
                        if not img_url or img_url in downloaded_urls: continue

                        time.sleep(0.8)
                        img_res = requests.get(img_url, headers=HEADERS, timeout=15)

                        if img_res.status_code == 200 and is_valid_image(img_res):
                            ext = img_url.split('.')[-1].lower().split('?')[0]
                            if len(ext) > 4: ext = 'jpg'

                            filename = f"{country}_{title}_{count}.{ext}"
                            with open(os.path.join(country_dir, filename), 'wb') as f:
                                f.write(img_res.content)

                            downloaded_urls.add(img_url)
                            count += 1
                        elif img_res.status_code == 429:
                            print("⚠️ Bị chặn khi tải ảnh! Nghỉ 2 phút...")
                            time.sleep(120)
                            break
                    except: continue

                # Ghi log hoàn tất địa danh
                with open(LOG_FILE, 'a', encoding='utf-8') as f:
                    f.write(wiki_url + '\n')
                processed_urls.add(wiki_url)

                if count > 0: print(f"   ✅ Đã lấy {count} ảnh.")

            except Exception as e:
                print(f"   ❌ Lỗi: {e}")
                continue

if __name__ == "__main__":
    start_mining()

Mounted at /content/drive

📂 Đang xử lý file: data_1.json

📂 Đang xử lý file: data_10.json

📂 Đang xử lý file: data_100.json

📂 Đang xử lý file: data_101.json

📂 Đang xử lý file: data_102.json

📂 Đang xử lý file: data_103.json

📂 Đang xử lý file: data_104.json

📂 Đang xử lý file: data_105.json

📂 Đang xử lý file: data_106.json

📂 Đang xử lý file: data_107.json

📂 Đang xử lý file: data_108.json

📂 Đang xử lý file: data_109.json

📂 Đang xử lý file: data_11.json

📂 Đang xử lý file: data_110.json

📂 Đang xử lý file: data_111.json

📂 Đang xử lý file: data_112.json

📂 Đang xử lý file: data_113.json

📂 Đang xử lý file: data_114.json

📂 Đang xử lý file: data_115.json

📂 Đang xử lý file: data_116.json

📂 Đang xử lý file: data_117.json

📂 Đang xử lý file: data_118.json

📂 Đang xử lý file: data_119.json

📂 Đang xử lý file: data_12.json

📂 Đang xử lý file: data_120.json

📂 Đang xử lý file: data_121.json

📂 Đang xử lý file: data_122.json

📂 Đang xử lý file: data_123.json

📂 Đang xử lý file: data_12